# From Prediction to Decision: Cost Matrices, Thresholds, Optimization and Simulation

Notebook for **Chapter 12** of the *ds* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ds/ch12.html

Tip: change a number, predict the output, then run the cell again.

## 12.1 Cost Matrices, Expected Value and Choosing the Threshold

In [ ]:
from ds_common import *
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
customers, orders = make_data()
d = make_label_table(customers, orders)
past = orders[orders.order_ts < PRED_DATE]
g = past.groupby("customer_id").agg(last=("order_ts", "max"), f=("order_id", "count"))
d = d.merge(g, left_on="customer_id", right_index=True)
d["rec"] = (PRED_DATE - d["last"]).dt.days
d["pro"] = (d.plan == "pro").astype(int); d["ads"] = (d.channel == "ads").astype(int)
d["logf"] = np.log1p(d.f)
X = d[["rec", "logf", "pro", "ads"]].values; y = d.churn.values
idx = np.random.default_rng(0).permutation(len(d)); tr, te = idx[:1700], idx[1700:]
mu, sd = X[tr].mean(0), X[tr].std(0)
lr = LogisticRegression().fit((X[tr] - mu) / sd, y[tr])
p = lr.predict_proba((X[te] - mu) / sd)[:, 1]; yt = y[te]
print(len(tr), len(te), "churn rate test", yt.mean().round(3), "AUC", round(roc_auc_score(yt, p), 3))

cost, save, val = 100, 0.3, 1500
def profit(send):
    return (send & (yt == 1)).sum() * save * val - send.sum() * cost
for t in [0.0, 0.1, cost/(save*val), 0.4, 0.6, 0.8]:
    s = p >= t; print(round(t, 3), "sent", s.sum(), "profit", round(profit(s)))

## 12.2 Optimization: Allocating Scarce Resources

In [ ]:
from scipy.optimize import linprog
c = -np.array([3.0, 2.2, 1.5])                 # linprog minimises, so negate
res = linprog(c, A_ub=[[1, 1, 1]], b_ub=[100], bounds=[(0, 50), (0, 40), (10, 60)])
print("alloc", res.x.round(1), "profit", round(-res.fun, 1), "shadow price", round(-res.ineqlin.marginals[0], 3))
res2 = linprog(c, A_ub=[[1, 1, 1]], b_ub=[101], bounds=[(0, 50), (0, 40), (10, 60)])
print("budget + 1 ->", round(-res2.fun, 2))

alloc [50. 40. 10.] profit 253.0 shadow price 1.5
budget + 1 -> 254.5


In [ ]:
from scipy.optimize import milp, LinearConstraint, Bounds
rng = np.random.default_rng(1); n = 12
value = np.round(rng.uniform(100, 900, n)); w = rng.integers(5, 30, n)
print("values ", value.astype(int).tolist()); print("minutes", w.tolist())
r = milp(-value, constraints=LinearConstraint(w[None, :], 0, 60), integrality=np.ones(n), bounds=Bounds(0, 1))
sel = np.where(r.x > 0.5)[0]
print("optimal", sel.tolist(), int(value[sel].sum()), int(w[sel].sum()))
order = np.argsort(-value / w); tot = wt = 0; gs = []          # greedy by value per minute
for i in order:
    if wt + w[i] <= 60: gs.append(int(i)); wt += w[i]; tot += value[i]
print("greedy ", sorted(gs), int(tot), wt)

values  [509, 860, 215, 859, 349, 439, 762, 427, 540, 122, 703, 531]
minutes [25, 13, 16, 24, 8, 12, 8, 16, 29, 8, 14, 15]
optimal [1, 4, 6, 10, 11] 3205 58
greedy  [1, 4, 5, 6, 10] 3113 55


In [ ]:
from scipy.optimize import linear_sum_assignment
C = np.array([[9, 2, 7, 8], [6, 4, 3, 7], [5, 8, 1, 8], [7, 6, 9, 4]])
ri, ci = linear_sum_assignment(C); print(list(zip(ri.tolist(), ci.tolist())), int(C[ri, ci].sum()))

[(0, 1), (1, 0), (2, 2), (3, 3)] 13


## 12.3 Simulation: Monte Carlo, What-If and Sensitivity

In [ ]:
rng = np.random.default_rng(42); N = 100000; n_send, cost = 273, 100
cs = rng.beta(60, 40, N); sv = rng.beta(3, 7, N); vl = rng.lognormal(np.log(1400), 0.3, N)
profit = n_send * cs * sv * vl - n_send * cost
print("mean profit", profit.mean().round(0), "P(loss)", (profit < 0).mean().round(3))
print("5%, 50%, 95%", np.percentile(profit, [5, 50, 95]).round(0))

mean profit 44614.0 P(loss) 0.104
5%, 50%, 95% [ -7266.  36817. 123503.]


In [ ]:
def mm1(lam, mu, n=200000, seed=0):
    r = np.random.default_rng(seed)
    ia = r.exponential(1/lam, n); sv = r.exponential(1/mu, n)
    w = np.zeros(n)
    for i in range(1, n): w[i] = max(0, w[i-1] + sv[i-1] - ia[i])
    resp = w + sv
    return resp.mean(), np.percentile(resp, 95)
for rho in [0.5, 0.8, 0.9]:
    m, p95 = mm1(rho * 10, 10)
    print(rho, "sim mean", round(m, 3), "theory", round(0.1/(1-rho), 3), "sim p95", round(p95, 3))

0.5 sim mean 0.202 theory 0.2 sim p95 0.609 theory p95 0.599
0.8 sim mean 0.505 theory 0.5 sim p95 1.498 theory p95 1.498
0.9 sim mean 1.019 theory 1.0 sim p95 3.005 theory p95 2.996


## 12.4 Decisions Under Uncertainty, Bandits and Feedback Loops

In [ ]:
r = np.random.default_rng(0)
pa = r.beta(1 + 40, 1 + 960, 200000); pb = r.beta(1 + 55, 1 + 945, 200000)
print("P(B>A)", (pb > pa).mean().round(3))
print("expected loss choosing B", np.maximum(pa - pb, 0).mean().round(5),
      "choosing A", np.maximum(pb - pa, 0).mean().round(5))

P(B>A) 0.941
expected loss choosing B 0.00024
choosing A 0.01519


## Chapter recap

## Practice set

**1.** (Easy) Offer cost 50, success rate 25%, value 800. Break-even probability? **Answer:** 50 / (0.25 × 800) = 50 / 200 = 0.25.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ds/ch12.html